# Task 10 — Python Revenue Forecasting & Backtesting Pipeline

**Project:** NovaMart Financial Analytics & Revenue Forecasting  
**Phase:** Task 10 — Time-Series Revenue Forecasting  
**Target Variable:** Monthly Net Revenue (`net_revenue`)  
**Data Source:** `data/processed/fact_sales_processed.csv`  

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.forecast import build_monthly_dataset, run_rolling_origin_backtest, generate_forward_forecast

# 1. Build & Reconcile Dataset
m_df = build_monthly_dataset()
print("Monthly Time Series Dataset (24 Months):")
print(m_df.head())


## 1. Rolling-Origin Backtesting & Accuracy Comparison (12 Evaluation Origins)


In [ ]:
metrics_df, models_dict, backtest_preds_df = run_rolling_origin_backtest(m_df)
print("=== MODEL ACCURACY COMPARISON TABLE ===")
print(metrics_df.to_string(index=False))

champion_name = metrics_df.iloc[0]['Model']
print(f"\nCHAMPION MODEL SELECTED: {champion_name} (Lowest WAPE: {metrics_df.iloc[0]['WAPE (%)']:.2f}%)")


## 2. Forward 6-Month Net Revenue Forecast (H1 2026)


In [ ]:
eval_actuals = m_df['net_revenue'].values[12:]
fc_df = generate_forward_forecast(m_df, champion_name, models_dict, eval_actuals)

print("=== FORWARD 6-MONTH REVENUE FORECAST (H1 2026) ===")
print(fc_df.to_string(index=False))
print(f"\nTOTAL H1 2026 PREDICTED NET REVENUE: ${fc_df['predicted_net_revenue'].sum():,.2f}")


## 3. Forecast Visualization Plots


In [ ]:
# Plot 1: Historical Actuals + 6-Month Forward Forecast
plt.figure(figsize=(12, 5))
plt.plot(m_df['month'], m_df['net_revenue'] / 1e6, marker='o', label='Historical Actuals', color='#1f77b4', linewidth=2)
plt.plot(fc_df['forecast_month'], fc_df['predicted_net_revenue'] / 1e6, marker='s', label=f'6-Month Forecast ({champion_name})', color='#ff7f0e', linewidth=2, linestyle='--')
plt.fill_between(fc_df['forecast_month'], fc_df['lower_bound_95'] / 1e6, fc_df['upper_bound_95'] / 1e6, color='#ff7f0e', alpha=0.2, label='95% Prediction Interval')
all_months = list(m_df['month']) + list(fc_df['forecast_month'])
plt.xticks(ticks=range(len(all_months)), labels=all_months, rotation=45, ha='right')
plt.title('NovaMart Monthly Net Revenue Forecast (H1 2026)', fontsize=13, fontweight='bold')
plt.ylabel('Net Revenue ($ Millions)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()
